# Transfer test: does low conversion cost mean easier adaptation?

**No ε, no disjoint control, no N\*.** Those are dropped. This notebook asks one direct question on the Exp6
grammar: if converting L_A → L_g is cheap (low translator cost `a(g)`), does a model trained on L_A
**adapt faster** to L_g than a model trained from scratch?

## Design

1. **Pretrain** a 12-layer, 12-head, d=96 rotary GPT on L_A only (fixed step budget, no early stopping).
2. **Adapt** copies of it to L_B and to L_C separately (fixed number of fine-tune steps, fresh optimizer),
   logging held-out CE on the target language every `EVAL_EVERY` steps.
3. **Scratch baseline:** a randomly initialised model trained on the same L_g data for the same steps.
4. **Savings:** `S_g = 1 − AUC_finetune(g) / AUC_scratch(g)`, where AUC is the mean held-out CE over the
   adaptation window (step 0 excluded; zero-shot CE is reported separately). Dividing by the scratch
   curve removes each language's own intrinsic difficulty.

Pretraining trees, fine-tuning trees and held-out trees come from three **different** tree samples (no
tree seen in L_A order is later reused). Within a seed, L_B and L_C are fine-tuned on the **same trees**.

**Tag handling (the `relabel_new` lesson):** the pretrained model never saw the BOS_B / BOS_C tags, so
their embedding and output rows are untrained. At fine-tune start those rows are **copied from BOS_A**, so
the new tag is not a cold-start confound. Both languages get the same treatment.

## Frozen BEFORE any transfer run

- The prediction comes from the translator notebook's `a(g)`, read and written to `prediction.json` by the freeze cell.
  The runs refuse to start without it, and the file is never overwritten.
- Higher `a(g)` = harder conversion = **less** savings. So `a(L_C) > a(L_B)` in every translator seed predicts
  `S_B > S_C`; the reverse predicts `S_C > S_B`; mixed signs predict **nothing** (the test then has nothing to test).
- **PASS** = predicted ordering of `S` holds in **every** seed. **FAIL** = reverse ordering in every seed.
  Otherwise INCONCLUSIVE. PROVISIONAL if fewer than 3 seeds.
- No seed-hunting and no rule changes after seeing numbers.

## What this can and can't show

- Adaptation speed is partly SGD speed. This is a behavioural readout of reuse, not a circuit measurement.
- Generic benefit (token statistics, positional machinery learned on L_A) helps both languages; it cancels in the
  B-vs-C comparison but not in the absolute savings numbers.
- Three languages means one binary comparison (B vs C). A PASS is mild support; a FAIL is informative.

In [1]:
import os, json, time, random, contextlib
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

# ================================ CONFIG ================================
QUICK_TEST = False        # True = tiny shakedown (writes to a separate folder, uses a dummy prediction). Set False for the real run.

DEVICE_STR = "cuda:2"    # check nvidia-smi before assuming this is free
DEVICE = torch.device(DEVICE_STR if torch.cuda.is_available() else "cpu")

D, N_LAYER, N_HEAD = 96, 12, 12
SEQ_LEN = 256
BATCH, LR = 32, 3e-3
PRE_STEPS, FT_STEPS, EVAL_EVERY = 40_000, 6_000, 250
PRE_TREES, FT_TREES, EVAL_TREES = 50_000, 30_000, 1_000
SEEDS = [0, 1, 2]

ROOT = "./exp6_transfer"
TRANSLATOR_ROOT = "./exp6_conv_cost"     # where the translator notebook wrote its JSONs
TRANSLATOR_D = 96
TRANSLATOR_SEEDS = [0, 1, 2]

if QUICK_TEST:
    D, N_LAYER, N_HEAD = 16, 2, 2
    PRE_STEPS, FT_STEPS, EVAL_EVERY = 600, 300, 100
    PRE_TREES, FT_TREES, EVAL_TREES = 2_000, 2_000, 100
    SEEDS = [0]
    ROOT = "./exp6_transfer_QUICK"
    TRANSLATOR_ROOT = "./exp6_conv_cost_QUICK"
    TRANSLATOR_D = 16
    TRANSLATOR_SEEDS = [0]

# Exp6 grammar settings (unchanged)
LEVELS, NT_SIZE, TERM_SIZE = 6, 4, 4
MIN_DEGREE, MAX_DEGREE, RULE_LEN_CHOICES = 2, 4, [2, 3]
GRAMMAR_SEED = 0

LANGS = {"L_A": set(), "L_B": {4, 5}, "L_C": set(range(1, LEVELS + 1))}
BOS = {"L_A": TERM_SIZE, "L_B": TERM_SIZE + 1, "L_C": TERM_SIZE + 2}   # same layout as Exp6
VOCAB = TERM_SIZE + len(LANGS)

os.makedirs(f"{ROOT}/adapt", exist_ok=True)
print(f"device={DEVICE}  QUICK_TEST={QUICK_TEST}  d={D} layers={N_LAYER} heads={N_HEAD}  "
      f"pre={PRE_STEPS} ft={FT_STEPS} seeds={SEEDS}")

device=cuda:2  QUICK_TEST=False  d=96 layers=12 heads=12  pre=40000 ft=6000 seeds=[0, 1, 2]


## Grammar (copied unchanged from Exp6)

In [2]:
from collections import defaultdict

def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1):
        symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE))
    rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE)
            seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1
                rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen:
                    continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol])
    children = []
    for child_sym in rhs:
        if level == LEVELS:
            children.append(child_sym)
        else:
            children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

def linearize(node, reverse_levels):
    if isinstance(node, int):
        return [node]
    (level, idx), children = node
    order = list(reversed(children)) if level in reverse_levels else children
    tokens = []
    for c in order:
        tokens.extend(linearize(c, reverse_levels))
    return tokens

rules, _ = build_grammar(GRAMMAR_SEED)
_t = sample_tree(rules, random.Random(123))
_a, _b, _c = (linearize(_t, LANGS[k]) for k in ("L_A", "L_B", "L_C"))
assert _c == list(reversed(_a)) and _b != _a and _b != _c
print("grammar sanity checks passed")

grammar sanity checks passed


## Data: token streams (Exp6 format)

One stream per language: `[BOS_l, tree tokens..., BOS_l, tree tokens, ...]`, chunked into `SEQ_LEN+1` windows.
Trees are sampled once and rendered under every language requested, so languages within a stream-set share trees.

In [3]:
_STREAM_CACHE = {}

def build_streams(langs, n_trees, data_seed):
    key = (tuple(langs), n_trees, data_seed)
    if key in _STREAM_CACHE:
        return _STREAM_CACHE[key]
    rng = random.Random(data_seed)
    toks = {l: [] for l in langs}
    for _ in range(n_trees):
        t = sample_tree(rules, rng)
        for l in langs:
            toks[l].append(BOS[l])
            toks[l].extend(linearize(t, LANGS[l]))
    out = {}
    for l in langs:
        s = torch.tensor(toks[l], dtype=torch.long)
        n_chunks = len(s) // (SEQ_LEN + 1)
        out[l] = s[: n_chunks * (SEQ_LEN + 1)].view(n_chunks, SEQ_LEN + 1)
    _STREAM_CACHE[key] = out
    return out

# distinct tree samples for the three roles
def pre_streams(seed):  return build_streams(("L_A",), PRE_TREES, 10_000 + seed)
def ft_streams(seed):   return build_streams(("L_B", "L_C"), FT_TREES, 20_000 + seed)
def eval_streams():     return build_streams(("L_A", "L_B", "L_C"), EVAL_TREES, 777_777)

_e = eval_streams()
print({k: tuple(v.shape) for k, v in _e.items()}, "(held-out chunks per language)")

{'L_A': (937, 257), 'L_B': (937, 257), 'L_C': (937, 257)} (held-out chunks per language)


## Model (copied unchanged from Exp6)

In [4]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=1024, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        cos, sin = self.rotary(T)
        q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, max_seq_len):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyRotaryGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_layer=12, n_head=12, max_seq_len=512):
        super().__init__()
        assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
    def forward(self, idx, targets=None):
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

def new_model():
    return TinyRotaryGPT(VOCAB, D, N_LAYER, N_HEAD, SEQ_LEN + 1).to(DEVICE)

_m = new_model()
print(f"model OK, {sum(p.numel() for p in _m.parameters())/1e6:.2f}M params at d={D}")
del _m

model OK, 1.34M params at d=96


## Training and evaluation

Fixed step budgets everywhere (no convergence flags, matched step counts across conditions). Evaluation is
mean CE over **all** positions of a held-out stream of the target language, the same quantity the training loss measures.

In [5]:
def autocast_ctx():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.bfloat16) if DEVICE.type == "cuda" else contextlib.nullcontext()

@torch.no_grad()
def eval_ce(model, data):
    model.eval()
    tot, n = 0.0, 0
    for i in range(0, data.shape[0], 64):
        b = data[i:i + 64].to(DEVICE)
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        tot += loss.item() * b.shape[0]; n += b.shape[0]
    model.train()
    return tot / n

def run_steps(model, data, steps, seed, eval_data=None, eval_every=None, tag=""):
    """AdamW, fixed budget. If eval_data is given, returns [(step, held-out CE)] incl. step 0."""
    torch.manual_seed(seed)
    data = data.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    curve = [(0, eval_ce(model, eval_data))] if eval_data is not None else []
    losses, t0 = [], time.time()
    for step in range(steps):
        idx = torch.randint(0, data.shape[0], (BATCH,), device=DEVICE)
        b = data[idx]
        x, y = b[:, :-1], b[:, 1:]
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step == 199:
            dt = (time.time() - t0) / 200
            print(f"  [{tag}] ~{dt*1000:.0f} ms/step -> projected {dt*steps/60:.0f} min for this stage", flush=True)
        if eval_data is not None and (step + 1) % eval_every == 0:
            curve.append((step + 1, eval_ce(model, eval_data)))
        if (step + 1) % 5000 == 0:
            print(f"  [{tag}] step {step+1:>6d} train loss (last 5000) {np.mean(losses[-5000:]):.4f}", flush=True)
    return curve

def get_pretrained(seed):
    """Pretrain on L_A only (cached on disk). Returns (state_dict, held-out L_A CE)."""
    path = f"{ROOT}/pretrained_L_A_s{seed}.pt"
    if os.path.exists(path):
        ck = torch.load(path, map_location="cpu", weights_only=False)
        print("pretrained model exists:", path)
        return ck["state"], ck["ce_A"]
    torch.manual_seed(seed)
    model = new_model()
    run_steps(model, pre_streams(seed)["L_A"], PRE_STEPS, seed, tag=f"pretrain s{seed}")
    ce_A = eval_ce(model, eval_streams()["L_A"].to(DEVICE))
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    torch.save({"state": state, "ce_A": ce_A, "steps": PRE_STEPS}, path)
    print(f"pretrained s{seed}: held-out L_A CE {ce_A:.4f}")
    return state, ce_A

def adapt(g, seed, init):
    """init='pretrained' (from L_A model, tag rows copied from BOS_A) or 'scratch'."""
    path = f"{ROOT}/adapt/{init}_{g}_s{seed}.json"
    if os.path.exists(path):
        print("done already:", path)
        return json.load(open(path))
    ev = eval_streams()[g].to(DEVICE)
    if init == "pretrained":
        state, ce_A = get_pretrained(seed)
        model = new_model()
        model.load_state_dict(state)
        with torch.no_grad():     # untrained tag rows would be a cold-start confound (relabel_new lesson)
            model.tok_emb.weight[BOS[g]] = model.tok_emb.weight[BOS["L_A"]].clone()
            model.head.weight[BOS[g]] = model.head.weight[BOS["L_A"]].clone()
    else:
        torch.manual_seed(seed + 1000)
        model = new_model()
    curve = run_steps(model, ft_streams(seed)[g], FT_STEPS, seed, eval_data=ev,
                      eval_every=EVAL_EVERY, tag=f"{init} {g} s{seed}")
    res = dict(init=init, g=g, seed=seed, ft_steps=FT_STEPS, eval_every=EVAL_EVERY, curve=curve)
    if init == "pretrained":
        res["pretrain_ce_A"] = ce_A
    json.dump(res, open(path, "w"))
    print(f"  {init} {g} s{seed}: zero-shot CE {curve[0][1]:.4f}  final CE {curve[-1][1]:.4f}")
    return res

## Freeze the prediction (from the translator notebook, before any transfer run)

Reads `a(g)` from the translator JSONs and writes `prediction.json`. If the file already exists it is
**loaded, not overwritten**. To change it you have to delete it by hand, which should only happen before any
transfer result has been seen. The run cell below refuses to start without it.

In [6]:
PRED_PATH = f"{ROOT}/prediction.json"

def _tr(g, s):
    p = f"{TRANSLATOR_ROOT}/translator/{g}_d{TRANSLATOR_D}_s{s}.json"
    return json.load(open(p)) if os.path.exists(p) else None

def _still_improving(curve):
    if len(curve) < 12:
        return None
    prev, last = np.mean(curve[-6:-3]), np.mean(curve[-3:])
    return bool(prev > 0.01 and (prev - last) > 0.10 * prev)

def freeze_prediction():
    if os.path.exists(PRED_PATH):
        P = json.load(open(PRED_PATH))
        print("prediction already frozen (loaded, not overwritten):")
        print(json.dumps({k: v for k, v in P.items() if k != "per_seed"}, indent=1))
        return P
    if QUICK_TEST:
        P = dict(prediction="S_B > S_C", note="QUICK_TEST dummy prediction, meaningless", frozen_at=time.ctime())
        json.dump(P, open(PRED_PATH, "w"))
        print("QUICK_TEST: wrote a dummy prediction")
        return P

    per_seed, flags = [], []
    for s in TRANSLATOR_SEEDS:
        R = {g: _tr(g, s) for g in ("identity", "L_B", "L_C")}
        assert all(R.values()), f"translator results missing for seed {s} in {TRANSLATOR_ROOT}; finish the translator notebook first"
        if R["identity"]["heldout_acc"] < 0.99:
            raise RuntimeError(f"translator instrument invalid at seed {s} (identity acc {R['identity']['heldout_acc']:.4f} < 0.99); "
                               "no prediction can be frozen from an invalid a(g)")
        aB = R["L_B"]["auc"] - R["identity"]["auc"]
        aC = R["L_C"]["auc"] - R["identity"]["auc"]
        per_seed.append(dict(seed=s, a_B=aB, a_C=aC))
        for g in ("L_B", "L_C"):
            if _still_improving(R[g]["curve"]):
                flags.append(f"translator {g} seed {s} still improving at the end")
    diffs = np.array([r["a_C"] - r["a_B"] for r in per_seed])
    if (diffs > 0).all():
        pred = "S_B > S_C"       # C costs more to convert -> less transfer savings
    elif (diffs < 0).all():
        pred = "S_C > S_B"
    else:
        pred = "none"            # sign not consistent across translator seeds
    P = dict(prediction=pred, a_C_minus_a_B_per_seed=diffs.tolist(), per_seed=per_seed,
             translator_flags=flags, frozen_at=time.ctime(), source=TRANSLATOR_ROOT)
    json.dump(P, open(PRED_PATH, "w"))
    print("FROZEN:", json.dumps({k: v for k, v in P.items() if k != "per_seed"}, indent=1))
    if flags:
        print("\nWARNING: translator curves were still improving, so a(g) may reflect the step budget. "
              "If you want to fix that, delete prediction.json and rerun the translators longer BEFORE running transfer.")
    if pred == "none":
        print("\nNo ordering is predicted (translator sign differs across seeds). Transfer can still be run descriptively.")
    return P

PRED = freeze_prediction()

prediction already frozen (loaded, not overwritten):
{
 "prediction": "S_C > S_B",
 "a_C_minus_a_B_per_seed": [
  -0.045735300324646275,
  -0.07252961711284567,
  -0.06583174516798865
 ],
 "translator_flags": [],
 "frozen_at": "Sun Nov  1 19:04:01 2026",
 "source": "./exp6_conv_cost"
}


## Run

Per seed: pretrain on L_A (cached), then adapt to L_B and L_C from the pretrained model, then train both from scratch.
Check the printed `ms/step` projection on the first stage before leaving it running. With `QUICK_TEST = True` this is a shakedown only.

In [7]:
assert os.path.exists(PRED_PATH), "run the freeze cell first: no transfer run without a frozen prediction"
for seed in SEEDS:
    for g in ("L_B", "L_C"):
        adapt(g, seed, "pretrained")
    for g in ("L_B", "L_C"):
        adapt(g, seed, "scratch")
print("\nall requested runs finished")

  [pretrain s0] ~20 ms/step -> projected 13 min for this stage
  [pretrain s0] step   5000 train loss (last 5000) 0.8355
  [pretrain s0] step  10000 train loss (last 5000) 0.8014
  [pretrain s0] step  15000 train loss (last 5000) 0.7943
  [pretrain s0] step  20000 train loss (last 5000) 0.7907
  [pretrain s0] step  25000 train loss (last 5000) 0.7881
  [pretrain s0] step  30000 train loss (last 5000) 0.7862
  [pretrain s0] step  35000 train loss (last 5000) 0.7847
  [pretrain s0] step  40000 train loss (last 5000) 0.7830
pretrained s0: held-out L_A CE 0.7914
  [pretrained L_B s0] ~18 ms/step -> projected 2 min for this stage
  [pretrained L_B s0] step   5000 train loss (last 5000) 0.7980
  pretrained L_B s0: zero-shot CE 1.6476  final CE 0.7944
pretrained model exists: ./exp6_transfer/pretrained_L_A_s0.pt
  [pretrained L_C s0] ~16 ms/step -> projected 2 min for this stage
  [pretrained L_C s0] step   5000 train loss (last 5000) 0.8125
  pretrained L_C s0: zero-shot CE 3.0775  final CE 

## Verdict (applies the rule frozen at the top)

In [9]:
def _load(init, g, s):
    p = f"{ROOT}/adapt/{init}_{g}_s{s}.json"
    return json.load(open(p)) if os.path.exists(p) else None

def _auc(curve):          # mean held-out CE over the adaptation window, step 0 excluded
    return float(np.mean([ce for st, ce in curve if st > 0]))

P = json.load(open(PRED_PATH))
seeds_done = [s for s in SEEDS if all(_load(i, g, s) for i in ("pretrained", "scratch") for g in ("L_B", "L_C"))]
print(f"seeds with all four runs: {seeds_done}")
assert seeds_done, "no complete seed yet, run the cell above first"

rows, flags = [], []
print(f"\n{'seed':>4} {'g':>4} {'zero-shot':>10} {'AUC_ft':>8} {'AUC_scr':>8} {'S_g':>7} {'final_ft':>9} {'final_scr':>10}")
for s in seeds_done:
    S, fin_scr = {}, {}
    for g in ("L_B", "L_C"):
        ft, sc = _load("pretrained", g, s), _load("scratch", g, s)
        a_ft, a_sc = _auc(ft["curve"]), _auc(sc["curve"])
        S[g] = 1.0 - a_ft / a_sc
        fin_scr[g] = sc["curve"][-1][1]
        print(f"{s:>4} {g:>4} {ft['curve'][0][1]:10.4f} {a_ft:8.4f} {a_sc:8.4f} {S[g]:7.3f} {ft['curve'][-1][1]:9.4f} {fin_scr[g]:10.4f}")
        if S[g] <= 0:
            flags.append(f"seed {s} {g}: S_g <= 0 (pretraining on L_A did not speed adaptation; negative transfer)")
    ce_A = _load("pretrained", "L_B", s).get("pretrain_ce_A")
    if ce_A is not None and ce_A > 0.9:
        flags.append(f"seed {s}: pretrained L_A held-out CE {ce_A:.3f} > 0.9 (L_A model poorly trained)")
    par = abs(fin_scr["L_B"] - fin_scr["L_C"]) / min(fin_scr.values())
    if par > 0.10:
        flags.append(f"seed {s}: scratch final CE differs between L_B and L_C by {par:.0%} (intrinsic difficulty differs; S_g normalises this but check)")
    rows.append(dict(seed=s, dS=S["L_C"] - S["L_B"], S=S))

dS = np.array([r["dS"] for r in rows])
print(f"\nS_C - S_B per seed: {np.round(dS, 4).tolist()}")
if len(dS) > 1:
    print(f"mean {dS.mean():.4f}  SE {dS.std(ddof=1)/np.sqrt(len(dS)):.4f}  (n={len(dS)})")
print("frozen prediction:", P["prediction"], "| a_C - a_B per translator seed:", P.get("a_C_minus_a_B_per_seed"))

pred = P["prediction"]
if pred == "none":
    verdict = "NO PREDICTION FROZEN (translator ordering not consistent across seeds); read the S ordering descriptively"
else:
    want_C_greater = (pred == "S_C > S_B")
    if (dS > 0).all():  verdict = "PASS" if want_C_greater else "FAIL"
    elif (dS < 0).all(): verdict = "FAIL" if want_C_greater else "PASS"
    else:               verdict = "INCONCLUSIVE (sign differs across seeds)"
if P.get("translator_flags"):
    verdict += " -- note: translator curves were flagged as still improving when the prediction was frozen"
if QUICK_TEST:
    verdict += " -- QUICK_TEST shakedown, dummy prediction, ignore"
elif len(rows) < 3:
    verdict += f" -- PROVISIONAL (only {len(rows)} seed(s); the rule needs 3)"

print("\nflags:", flags if flags else "none")
print("\nVERDICT:", verdict)

seeds with all four runs: [0, 1, 2]

seed    g  zero-shot   AUC_ft  AUC_scr     S_g  final_ft  final_scr
   0  L_B     1.6476   0.7977   0.8250   0.033    0.7944     0.8018
   0  L_C     3.0775   0.8039   0.8216   0.022    0.7949     0.8029
   1  L_B     1.6454   0.7966   0.8255   0.035    0.7929     0.8028
   1  L_C     3.0986   0.8040   0.8204   0.020    0.7945     0.8017
   2  L_B     1.6292   0.7979   0.8163   0.023    0.7948     0.7978
   2  L_C     3.1492   0.8054   0.8210   0.019    0.7958     0.8002

S_C - S_B per seed: [-0.0114, -0.0151, -0.0036]
mean -0.0100  SE 0.0034  (n=3)
frozen prediction: S_C > S_B | a_C - a_B per translator seed: [-0.045735300324646275, -0.07252961711284567, -0.06583174516798865]

flags: none

VERDICT: FAIL


## Reading it

- **PASS:** the language that is cheaper to convert from L_A also adapts faster from an L_A-trained model, in every seed. Mild support (one binary comparison).
- **FAIL:** the reverse ordering in every seed. Both numbers are measured where they are valid, so this can't be blamed on imprecision.
- **S_g ≤ 0:** pretraining on L_A doesn't help adaptation at all. If that holds for both, the model isn't reusing its L_A machinery for L_B/L_C in any way this readout can see.
- Zero-shot CE (step 0) shows how much an L_A model already handles the other orderings before any adaptation.
- To turn this into a correlation you need more linearizations. Once the translator notebook's `LANGS` gets more entries, write the predicted `S` ordering from their `a(g)` first, then add them here.